# Hˢ Kinematics Engine — Replication Notebook (v1.0)

*Reproduce the full Hˢ kinematics stack on one composition trajectory, and verify the determinism receipt.*

This notebook runs the **unified engine** (`hs_kinematics_engine.run`) and the **diagnosis language**
(`hs_diagnosis.diagnose`) on a fixed reference, and checks the result against the conformance hash in
`HS_KINEMATICS_SPECIFICATION.md` §11. numpy + stdlib only. Author: Peter Higgins; AI-assisted per HUF-STD-001.
Honest-broker; Tier 1 except the fringe layer (Tier 3).


## 1. Imports

The engine and diagnosis modules live alongside this notebook.


In [ ]:
import numpy as np, json
import hs_kinematics_engine as eng
import hs_diagnosis as dx


## 2. The reference composition

A fixed 12×6 energy-transition matrix — hardcoded so it is byte-for-byte reproducible on any platform
(no RNG). Coal/Gas decline; Wind/Solar rise. This is the conformance reference.


In [ ]:
names = ['Coal','Gas','Hydro','Nuclear','Wind','Solar']
ref = np.array([[40,25,15,12,5,3],[38,25,15,12,6,4],[35,25,16,12,8,4],[33,24,16,12,9,6],
                [30,24,16,13,10,7],[28,23,17,13,11,8],[25,23,17,13,13,9],[22,22,18,13,14,11],
                [20,22,18,14,15,11],[18,21,18,14,17,12],[16,21,19,14,18,12],[15,20,19,14,19,13]], float)
ref.shape


## 3. Run the engine

One call returns the entire state: lossless reconstruction, navigation reads,
the kinematic/dynamic tower, spectral modes, the honesty guards, the fringe/boundary test, the floors, and the hash.


In [ ]:
out = eng.run(ref, names)
print(json.dumps(out, indent=1))


## 4. Read the headline quantities

Each is named twice (navigation / physics).


In [ ]:
rec = out['dead_reckoning_NAV__lossless_reconstruction_PHYS']
k   = out['kinematics_and_dynamics']
print('lossless exact      :', rec['exact'], ' error =', rec['reconstruction_error'])
print('effective dimension :', out['spectral_modes']['degrees_of_freedom_NAV__effective_dimensionality_PHYS'])
print('max derivative order:', out['computational_floors']['max_meaningful_derivative_order'])
print('discovered noise floor:', out['station_keeping_NAV__equilibrium_hold_PHYS']['discovered_noise_floor'])
print('arrow of intent  to :', k['arrow_of_intent_NAV__momentum_PHYS']['to'])
print('arrow of intent from:', k['arrow_of_intent_NAV__momentum_PHYS']['from'])
print('coherence (1=ballistic,0=churn):', k['arrow_of_intent_NAV__momentum_PHYS']['coherence'])
print('course directness/path efficiency:', k['course_directness_NAV__path_efficiency_PHYS'])
print('fringe verdict (Tier 3):', out['fringe_boundary_TIER3']['verdict'])
print('guards fired:', out['guards_codes_fired'])


## 5. The diagnosis language

The same readings, spoken. The number of active *voices* scales with the number of carriers actually moving.


In [ ]:
d = dx.diagnose(ref, names)
print(d['narrative'])
print('active voices:', d['active_voices'])


## 6. Determinism — the conformance receipt

Same data → same payload → same hash. We run twice and check identity, then compare to the spec §11 anchor.


In [ ]:
h1 = eng.run(ref, names)['content_hash']
h2 = eng.run(ref, names)['content_hash']
ANCHOR = 'fcae0ebe5c4f443aa076d1900d3d04219c2628591323cd7745621e740a3d7ae7'
print('hash:', h1)
print('deterministic (h1==h2):', h1==h2)
print('matches spec §11 anchor:', h1==ANCHOR)
assert h1==h2==ANCHOR, 'CONFORMANCE FAIL — a located deviation to explain (see ADAPTIVE_ANTICIPATION.md)'
print('CONFORMANT ✓')


## 7. Try your own data

Make any data zip / CSV / xlsx engine-ready with `hs_data_prep.py` (see `DATA_PREP.md`), then `eng.run(M, names)`.
The engine holds or warns rather than emit a confident-wrong reading; the meaning of the numbers stays yours.

*The geometry is CoDa's; the engine is the instrument built on it.*
